In [ ]:
from pathlib import Path
import numpy as np
from IORlib.Flow import Flow

input = np.load('ekofisk_cl.npz', allow_pickle=True)
#input = np.load('ekofisk_tor-eko-wi.npz', allow_pickle=True)
#input = np.load('simple_model-cl.npz', allow_pickle=True)

root = str(input['root'])
#cop = Path.home() / 'CoP_files'
#root = cop/'input/IORSIM_BASECASE_PYSIM/IORSIM_BASECASE_INPUT'
#root = cop/'input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'

print('Root:', root)
flow = Flow(root, res_block=True)

#tracer_func = flow.tracer_explicit
tracer_func = flow.tracer_implicit
tracer = tracer_func(input['name'], input['init'], input['inlet'], 
                     force_vol_balance=False)

# Step through time
stop = None
n_write = 0
report_time = 1e-6
try:
    for n, trc in enumerate(tracer):
        print('--', trc)
        print('   ' + trc.limits())
        trc.save_mass()
        trc.save_conc()
        if report_time and int(trc.time/report_time) > n_write:
            file = trc.to_unrst()
            n_write += 1
            print(f'   Data written to {file}')
        if n + 1 == stop:
            break
except KeyboardInterrupt:
    print('Simulation cancelled')



Root: /home/AD.NORCERESEARCH.NO/javi/CoP_files/input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT
-- step:    1, time:     0.010 days, dt:  0.010 days, cycles: 1
   CL: (1.420, 3.330)
Writing mass to IORSIM_BASECASE_INPUT_tracer_mass.dat
Writing well concentrations to ['IORSIM_BASECASE_INPUT_tracer_conc_CL.dat']
Creating new UNRST-file: IORSIM_BASECASE_INPUT_tracer.UNRST
   Data written to IORSIM_BASECASE_INPUT_tracer.UNRST
-- step:    2, time:     0.040 days, dt:  0.030 days, cycles: 1
   CL: (1.420, 3.331)
   Data written to IORSIM_BASECASE_INPUT_tracer.UNRST
-- step:    3, time:     0.130 days, dt:  0.090 days, cycles: 1
   CL: (1.419, 3.331)
   Data written to IORSIM_BASECASE_INPUT_tracer.UNRST


In [5]:
arr = np.zeros(10)
arr.ndim

1

In [ ]:
from pathlib import Path
import numpy as np
from IORlib.Flow import unrstfile

cop = Path.home() / 'CoP_files'
#root = cop/'input/IORSIM_BASECASE_PYSIM/IORSIM_BASECASE_INPUT'
root = cop/'input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'

unrst = unrstfile(root)
def find_nan_block(unrst):
    for seq, time, data in unrst.blockdata('SEQNUM', 0, 'DOUBHEAD', 0, 'CL'):
        cl, = unrst.reshape_dim(data)
        isnan = np.isnan(cl)
        if np.any(isnan):
            return (seq, time, np.argwhere(isnan)[0])
        
seq, time, nanblock = find_nan_block(unrst)
nanblock

In [ ]:
from IORlib.Flow import Flow
flow = Flow(root, res_block=True)
nnc = flow.nnc.nnc
np.argwhere(np.all(nnc[0] == nanblock, axis=1))

nnc[0][28461]


In [ ]:
# Input Ekofisk Chlorine tracer
from pathlib import Path
import numpy as np
from IORlib.ECL import INIT_file

cop = Path.home() / 'CoP_files'
root = cop/'input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'

# Tracer name
name = ('CL',)

# Initial coonditions
dim = INIT_file(root).dim()
cl = np.zeros(dim)
# Tor formation
cl[:,:,13:] = 3.33
# Ekofisk formation
cl[:,:,:13] = 1.42
init = (cl,)

# Inlet conditions
# Seawater
inlet = (0.56,)

np.savez('ekofisk_cl.npz', root=root, name=name, init=init, inlet=inlet)

In [ ]:
# Input Ekofisk TOR, EKO, WI tracers
from pathlib import Path
import numpy as np
from IORlib.ECL import INIT_file

cop = Path.home() / 'CoP_files'
root = cop/'input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'

# Tracer names
name = ('TOR', 'EKO', 'WI')

# Initial conditions
dim = INIT_file(root).dim()
tor = np.zeros(dim)
tor[:,:,14:] = 1
eko = np.zeros(dim)
eko[:,:,:14] = 1
wi = np.zeros(dim)
init = (tor, eko, wi)

# Inlet conditions
inlet = (0, 0, 1)

np.savez('ekofisk_tor-eko-wi.npz', root=root, name=name, init=init, inlet=inlet)

In [3]:
# Input simple model, inject WI = 1 in empty reservoir
from pathlib import Path
import numpy as np
from IORlib.ECL import INIT_file

cop = Path.home() / 'CoP_files'
root = cop/'input/Model_From_IXF/Model_From_IXF'

# Tracer name
name = ('WI',)

# Initial coonditions
dim = INIT_file(root).dim()
init = (np.zeros(dim),)

# Inlet conditions
inlet = (1,)

np.savez('simple_model-wi.npz', root=root, name=name, init=init, inlet=inlet)


In [2]:
# Input simple model, inject CL = 0.5 in 1.0 reservoir
from pathlib import Path
import numpy as np
from IORlib.ECL import INIT_file

cop = Path.home() / 'CoP_files'
root = cop/'input/Model_From_IXF/Model_From_IXF'

# Tracer name
name = ('CL',)

# Initial coonditions
dim = INIT_file(root).dim()
init = (np.ones(dim),)

# Inlet conditions
inlet = (0.5,)

np.savez('simple_model-cl.npz', root=root, name=name, init=init, inlet=inlet)
